<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Filtrage d'images

Les images contiennent une quantité considérable de données. Une image en niveaux de gris de $640 \times 480$ pixels comprend 307 200 pixels, chacun pouvant prendre l'une des 256 valeurs possibles. Analyser des images brutes pour estimer la structure de l'environnement d'un robot est une tâche complexe, surtout lorsqu'il s'agit de traiter des images en temps réel, captées à 30 images par seconde par la caméra du robot.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/image_filtering/image-filtering-block-diagram.png" width="400px" />
  <p>Le filtrage d'images effectue des opérations locales sur une image à l'aide d'une matrice de noyau afin d'en améliorer le contenu.</p>
  </div>
</figure>

Le filtrage d'images permet d'améliorer une image en amplifiant certains aspects souhaitables et en atténuant d'autres. Il en résulte une transformation de l'image brute originale en un ensemble plus concis d'informations pertinentes.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/image_filtering/image-filtering-local-window.png" width="400px" />
  <p>Le filtrage d'images effectue des opérations locales sur une image à l'aide d'une matrice de noyau afin d'en améliorer le contenu.</p>
  </div>
</figure>

En filtrage d'images, chaque pixel de l'image de sortie est calculé à partir d'une combinaison pondérée des pixels voisins de l'image d'entrée. Cette pondération est appelée *noyau de filtre* et prend la forme d'une petite matrice (relativement à l'image) $h$. Il est important de noter que le noyau est identique sur toute l'image.

Mathématiquement, le filtrage d'images peut être représenté comme la convolution de l'image avec le noyau de filtre :

$$Y = I*h$$

ce qui, pour chaque pixel $u,v$ de l'image de sortie, devient :

$$Y[u,v] = \sum_{k,l} h[k,l]I[u-k,v-l]$$

Remarquez que le terme $I[u-k,v-l]$ dans la somme inverse le noyau $h$ horizontalement et verticalement.

Le noyau détermine quel contenu est amplifié et quel contenu est supprimé ; nous concevons donc le noyau en fonction de la manière dont nous souhaitons améliorer l'image.

### Exemple : Brouiller une image (*blurring*)

Dans cet exemple, on nous donne une image bruitée et l'on souhaite concevoir un filtre qui élimine le maximum de bruit sans trop altérer les détails. Le filtre « boîte », une matrice normalisée de uns, est un candidat potentiel, car le bruit est de haute fréquence (variation très rapide) tandis que le contenu de l'image est principalement de basse fréquence.

$$ h = \frac{1}{N} 
\begin{bmatrix}
1 & 1 & \cdots & 1\\
1 & 1 & \cdots & 1\\
\vdots & \vdots & \ddots & \vdots\\
1 & 1 & \cdots & 1
\end{bmatrix}
$$

où $N$ représente le nombre d'éléments de la matrice. Un filtre de type boîte (*box filter*) conserve les basses fréquences tout en atténuant les hautes fréquences. Il y parvient en lissant la valeur de chaque pixel pour qu'elle corresponde à la moyenne des valeurs des pixels voisins.

Lors de la conception d'un filtre de boîte, nous pouvons choisir la largeur et la hauteur du noyau, en supposant toutefois que la matrice est carrée. Augmenter la taille du noyau signifie qu'un plus grand nombre de pixels voisins seront utilisés pour déterminer l'intensité moyenne à chaque point de sortie. Intuitivement, cela améliore la réduction du bruit, mais au détriment des détails de l'image.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
### Exécutez cette cellule pour importer les modules pertinents
from matplotlib import pyplot as plt
import numpy as np
import cv2

In [ ]:
# TODO: Créez un filtre de type boîte 3x3 et comparez le résultat de la convolution de l'image bruitée ci-dessus avec celui de filtres de type boîte plus larges.
# Pouvez-vous déterminer la largeur du filtre au-delà de laquelle la perte de détails est trop importante ?

width = 3
hbox = np.random.rand(width,width) # CHANGEZ-MOI

# charger l'image
img = cv2.imread('../../assets/images/image_filtering/zebra-noisy.jpg', 0)

# filtrer l'image source
img_box_filter = cv2.filter2D(img,-1,hbox)

# Visualisez l'image filtrée à côté de l'image originale.
fig = plt.figure(figsize = (20,20))
ax1 = fig.add_subplot(1,2,1)
ax1.imshow(img,cmap = 'gray')
ax1.set_title('Original'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(img_box_filter,cmap = 'gray')
ax2.set_title('Box Filter (Width = ' + str(width) +')'), ax2.set_xticks([]), ax2.set_yticks([]);

## Brouiller une image avec un Gaussien (*Gaussian blurring*) 

Bien que le filtre de boîte soit intuitif, son utilisation pour lisser une image avant le calcul des dérivées numériques peut engendrer des artefacts (pourquoi ?). En pratique, on utilise généralement un noyau qui approxime une distribution gaussienne isotrope (c'est-à-dire que les sections transversales sont circulaires, la distribution est uniformément répartie et les valeurs de la diagonale de la matrice de covariance sont identiques) de moyenne nulle (c'est-à-dire centrées à l'origine).

$$ G_\sigma = \frac{1}{2 \pi \sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}} $$

où $\sigma$ représente l'écart type. L'avantage d'utiliser un noyau gaussien plutôt qu'un filtre de type boîte est qu'il accorde plus d'importance aux voisins les plus proches qu'à ceux qui sont plus éloignés.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/image_filtering/gaussian-kernel.png" width="400px" />
  <p>Une gaussienne 2D (à gauche) et un noyau gaussien (à droite).</p>
  </div>
</figure>

L'écart type $\sigma$ détermine la vitesse de décroissance de la gaussienne (son étendue). Pour des valeurs élevées de $\sigma$, la gaussienne décroît lentement et, par conséquent, les pixels voisins ont un poids similaire. À l'inverse, de faibles valeurs de $\sigma$ produisent une distribution plus concentrée, avec des poids qui diminuent rapidement.

Bien que les gaussiennes aient un support infini, nous nous intéressons aux noyaux finis (et généralement petits). Le choix de la taille du noyau dépend généralement du support de la distribution (c'est-à-dire de son étendue), qui est déterminé par l'écart type. Lorsque l'écart type est faible et que la distribution est concentrée, un petit noyau suffit car les poids en dehors du noyau sont très faibles. En revanche, lorsque l'écart type est plus élevé et que la distribution est plus étendue, on a tendance à utiliser un noyau plus grand.

### Exemple

Le noyau gaussien est défini par sa taille et son variance $\sigma$. Le variance déterminant la vitesse de décroissance des poids du noyau, il est courant de choisir sa taille (carrée pour les gaussiennes isotropes) en fonction de cet variance. De nombreuses bibliothèques existantes, dont l'implémentation dans OpenCV, prennent en charge cette fonctionnalité.

Dans cet exercice, vous expérimenterez différentes valeurs de l'écart type et les comparerez au résultat obtenu avec le filtre de boîte identifié précédemment. Vous pouvez consulter la définition de [cv2.GaussianBlur](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html#gae8bdcd9154ed5ca3cbc1766d960f45c1) pour comprendre le fonctionnement de la fonction `GaussianBlur`.

Essayez de trouver un réglage pour variance qui réduise le bruit sans supprimer trop de contenu de l'image.

In [ ]:
# TODO: Expérimentez différents réglages de l'écart type du filtre gaussien et comparez les résultats à ceux d'un filtre de type boîte.
#. Trouvez un réglage de l'écart type qui élimine le bruit sans trop altérer le contenu de l'image.

sigma = 1

# Lisser l'image à l'aide d'un noyau gaussien

img_gaussian_filter = cv2.GaussianBlur(img,(0,0), sigma)

# Visualisez l'image filtrée à côté de l'image originale.
fig = plt.figure(figsize = (20,20))
ax1 = fig.add_subplot(1,3,1)
ax1.imshow(img,cmap = 'gray')
ax1.set_title('Original'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,3,2)
ax2.imshow(img_gaussian_filter,cmap = 'gray')
ax2.set_title('Gaussian Filter (Sigma = ' + str(sigma) +')'), ax2.set_xticks([]), ax2.set_yticks([]);
ax3 = fig.add_subplot(1,3,3)
ax3.imshow(img_box_filter,cmap = 'gray')
ax3.set_title('Box Filter'), ax3.set_xticks([]), ax3.set_yticks([]);

## Contours (*Edges*)

Les contours sont une caractéristique d'une image utilisée pour diverses tâches, notamment la segmentation, la détection d'objets, l'estimation de formes et la reconstruction de scènes. Dans le cas de Duckietown et d'autres applications de conduite autonome, l'accès aux informations de contour est également très utile pour la détection des limites de voies.

<figure>
  <div style="text-align:center;">
  <img src="../../assets/images/image_filtering/1d-intensity-change.png" width="700px" />
  <p>Le changement d'intensité associé à un contour où une image passe du blanc au noir et du noir au blanc.</p>
  </div>
</figure>

Les contours correspondent à des variations importantes d'intensité locale. Cela suggère que l'on peut identifier les contours en recherchant dans l'image les régions où le gradient d'intensité $\nabla I$ est élevé. Prenons l'exemple de l'image la plus à gauche ci-dessus, qui pourrait être un agrandissement d'une des rayures verticales d'un zèbre. En parcourant une ligne de l'image de gauche à droite, on observe que l'intensité chute rapidement lorsque les pixels passent du blanc (proche de 255) au noir (proche de 0), puis reste faible jusqu'à ce qu'elle remonte brusquement lorsque les pixels passent du noir au blanc. Les contours sont clairement visibles sous forme de pics dans la composante $x$ du gradient d'intensité.

De manière générale, nous nous intéressons au gradient bidimensionnel de l'image, qui inclut les variations d'intensité horizontales et verticales: 

$$ \nabla I[u,v] = 
\begin{bmatrix}
\frac{\partial I[u,v]}{\partial x}\\
\frac{\partial I[u,v]}{\partial y}
\end{bmatrix}
$$

L'amplitude du gradient $\lvert \nabla I[u,v] \rvert$ fournit une mesure de la force du contour, et la direction $\theta[u,v]$ nous indique la direction de la plus grande variation.

$$ 
\lvert \nabla I[u,v] \rvert = \sqrt{\left(\frac{\partial I[u,v]}{\partial x}\right)^2 + \left(\frac{\partial I[u,v]}{\partial y}\right)^2} \qquad
\theta[u,v] = \textrm{atan2}\left(\frac{\partial I[u,v]}{\partial y}, \frac{\partial I[u,v]}{\partial x}\right)
$$

Comme les pixels sont discrets, nous ne pouvons pas calculer les dérivées analytiquement. Nous les estimons donc numériquement par une approximation aux différences finies. Ces estimations sont obtenues en convoluant l'image avec des noyaux appropriés $h_x$ et $h_y$ qui calculent les dérivées finies selon les directions $x$ et $y$, respectivement.

$$ \nabla I[u,v] = 
\begin{bmatrix}
G_x[u,v]\\
G_y[u,v]
\end{bmatrix}
$$

où

$$ 
\begin{align}
G_x = h_x * I\\
G_y = h_y * I
\end{align}
$$

### Exemple

Dans cet exemple, vous allez concevoir différents noyaux de filtre, $h_x$ et $h_y$, pour estimer les gradients horizontal et vertical d'une image. N'oubliez pas que le filtre sera inversé horizontalement et verticalement lors de la convolution.

In [ ]:
# TODO: Concevez deux noyaux hx et hy pour estimer les gradients horizontal et vertical d'une image.
# Ces filtres seront appliqués à l'image floue que vous avez générée précédemment. Lors de vos essais avec différents
# noyaux, comparez les résultats pour différents niveaux de flou (c.-à-d. différents écarts-types). En particulier,
# comparez les gradients et leur amplitude avec un flou faible (c.-à-d. sigma=1) ou nul à ceux d'une image floutée avec un noyau de grande valeur

hx = np.array(np.random.rand(3,3), dtype=np.float32) # MODIFIEZ-MOI, MAIS CONSERVEZ dtype=np.float32
hy = np.array(np.random.rand(3,3), dtype=np.float32) # MODIFIEZ-MOI, MAIS CONSERVEZ dtype=np.float32

print(hx)

# Appliquez les filtres à l'image floue
Gx = cv2.filter2D(img_gaussian_filter.astype(np.float32),-1,hx)
Gy = cv2.filter2D(img_gaussian_filter.astype(np.float32),-1,hy)


# Calculer l'amplitude des gradients
Gmag = np.sqrt(Gx*Gx + Gy*Gy)

# Visualisez l'image filtrée à côté de l'image originale.
fig = plt.figure(figsize = (20,20))
ax1 = fig.add_subplot(1,4,1)
ax1.imshow(img,cmap = 'gray')
ax1.set_title('Blurred Image'), ax1.set_xticks([]), ax1.set_yticks([])
ax2 = fig.add_subplot(1,4,2)
ax2.imshow(Gx,cmap = 'gray')
ax2.set_title('Horizontal Gradients'), ax2.set_xticks([]), ax2.set_yticks([]);
ax3 = fig.add_subplot(1,4,3)
ax3.imshow(Gy,cmap = 'gray')
ax3.set_title('Vertical Gradients'), ax3.set_xticks([]), ax3.set_yticks([]);
ax4 = fig.add_subplot(1,4,4)
ax4.imshow(np.uint8(Gmag),cmap = 'gray')
ax4.set_title('Gradient Magnitude'), ax4.set_xticks([]), ax4.set_yticks([]);

### Sobel Operator
Un ensemble de noyaux populaire utilisé pour accentuer les contours d'une image est l'*opérateur de Sobel*, qui se compose de deux noyaux $3 \times 3$ :

$$ h_x = 
\begin{bmatrix}
+1 & 0 & -1\\
+2 & 0 & -2\\
+1 & 0 & -1
\end{bmatrix}\qquad
h_y = 
\begin{bmatrix}
+1 & +2 & +1\\
0 & 0 & 0\\
-1 & -2 & -1
\end{bmatrix}
$$

Ces noyaux correspondent à un flou simple basé sur la moyenne, suivi d'une dérivée finie simple :

$$ h_x = 
\begin{bmatrix}
+1 & 0 & -1\\
+2 & 0 & -2\\
+1 & 0 & -1
\end{bmatrix} = 
\begin{bmatrix}
1\\
2\\
1
\end{bmatrix} * 
\begin{bmatrix}
+1 & 0 & -1\\
\end{bmatrix} \qquad
h_y = 
\begin{bmatrix}
+1 & +2 & +1\\
0 & 0 & 0\\
-1 & -2 & -1
\end{bmatrix} = 
\begin{bmatrix}
+1\\
0\\
-1
\end{bmatrix} * 
\begin{bmatrix}
1 & 2 & 1\\
\end{bmatrix}
$$

### Exemple

Dans cet exemple, nous comparerons les gradients horizontaux et verticaux identifiés par le filtre que nous avons conçu avec ceux identifiés par l'opérateur de Sobel. Ce faisant, vous observerez comment le filtre que vous avez conçu et l'opérateur de Sobel réagissent différemment à différents niveaux de lissage.

**Remarque :** L'opérateur de Sobel applique un lissage par défaut, contrairement au filtre que vous avez conçu.

In [ ]:
sobelx = cv2.Sobel(img_gaussian_filter,cv2.CV_64F,1,0)
sobely = cv2.Sobel(img_gaussian_filter,cv2.CV_64F,0,1)

# Calculer l'amplitude des gradients basés sur la méthode de Sobel
sobelmag = np.sqrt(sobelx*sobelx + sobely*sobely)

fig = plt.figure(figsize = (20,10))
ax1 = fig.add_subplot(2,3,1)
ax1.imshow(Gx,cmap = 'gray')
ax1.set_title('Horizontal Gradients'), ax1.set_xticks([]), ax1.set_yticks([]);
ax2 = fig.add_subplot(2,3,2)
ax2.imshow(Gy,cmap = 'gray')
ax2.set_title('Vertical Gradients'), ax2.set_xticks([]), ax2.set_yticks([]);
ax3 = fig.add_subplot(2,3,3)
ax3.imshow(np.uint8(Gmag),cmap = 'gray')
ax3.set_title('Gradient Magnitude'), ax3.set_xticks([]), ax3.set_yticks([]);
ax4 = fig.add_subplot(2,3,4)
ax4.imshow(sobelx,cmap = 'gray')
ax4.set_title('Horizontal Gradients (Sobel)'), ax4.set_xticks([]), ax4.set_yticks([]);
ax5 = fig.add_subplot(2,3,5)
ax5.imshow(sobely,cmap = 'gray')
ax5.set_title('Vertical Gradients (Sobel)'), ax5.set_xticks([]), ax5.set_yticks([]);
ax6 = fig.add_subplot(2,3,6)
ax6.imshow(np.uint8(sobelmag),cmap = 'gray')
ax6.set_title('Gradient Magnitude (Sobel)'), ax6.set_xticks([]), ax6.set_yticks([]);

## Conclure

Dans ce notebook, nous avons exploré quelques filtres permettant d'améliorer une image, que ce soit en supprimant le bruit ou d'autres détails indésirables, ou en accentuant la présence d'éléments spécifiques, tels que les contours. Ces filtres, parmi d'autres, sont essentiels à l'extraction d'informations à partir d'images, qu'il s'agisse d'identifier l'emplacement du marquage au sol et, par conséquent, la géométrie de la voie du robot, ou de détecter la présence de piétons (par exemple, des canards). Ce ne sont là que quelques exemples de filtres couramment utilisés en traitement d'images. Fonctionnant selon le même principe que les filtres de détection de contours, les filtres de détection de coins recherchent dans l'image les zones de variations d'intensité rapides selon les axes x et y (comme aux angles où deux lignes se croisent). Ces filtres, et d'autres similaires, sont souvent utilisés pour détecter les « points d'intérêt », des points saillants de l'image permettant d'identifier les correspondances entre plusieurs images d'une même scène (c'est-à-dire les pixels correspondant au même objet dans l'espace réel).

Vous pouvez maintenant passer au [notebook sur contrôle visuomoteur (*visual servoing*)](../04-Visual-Servoing/visual_servoing.ipynb).